In [1]:
print("Here begins the hybrid models notebook")

Here begins the hybrid models notebook


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
df_ml = pd.read_csv("../../data/processed/mental_health_tech_ml.csv")

df_ml.head()

,Age,self_employed,family_history,treatment,remote_work,tech_company,obs_consequence,Age__missing,Gender__missing,state__missing,self_employed__missing,work_interfere__missing,Gender_male,Country_Austria,"Country_Bahamas, The",Country_Belgium,Country_Bosnia and Herzegovina,Country_Brazil,Country_Bulgaria,Country_Canada,Country_China,Country_Colombia,Country_Costa Rica,Country_Croatia,Country_Czech Republic,Country_Denmark,Country_Finland,Country_France,Country_Georgia,Country_Germany,Country_Greece,Country_Hungary,Country_India,Country_Ireland,Country_Israel,Country_Italy,Country_Japan,Country_Latvia,Country_Mexico,Country_Moldova,Country_Netherlands,Country_New Zealand,Country_Nigeria,Country_Norway,Country_Philippines,Country_Poland,Country_Portugal,Country_Romania,Country_Russia,Country_Singapore,Country_Slovenia,Country_South Africa,Country_Spain,Country_Sweden,Country_Switzerland,Country_Thailand,Country_United Kingdom,Country_United States,Country_Uruguay,Country_Zimbabwe,state_AZ,state_CA,state_CO,state_CT,state_DC,state_FL,state_GA,state_IA,state_ID,state_IL,state_IN,state_KS,state_KY,state_LA,state_MA,state_MD,state_ME,state_MI,state_MN,state_MO,state_MS,state_NC,state_NE,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OH,state_OK,state_OR,state_PA,state_RI,state_SC,state_SD,state_TN,state_TX,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY,work_interfere_Often,work_interfere_Rarely,work_interfere_Sometimes,no_employees_100-500,no_employees_26-100,no_employees_500-1000,no_employees_6-25,no_employees_More than 1000,benefits_No,benefits_Yes,care_options_Not sure,care_options_Yes,wellness_program_No,wellness_program_Yes,seek_help_No,seek_help_Yes,anonymity_No,anonymity_Yes,leave_Somewhat difficult,leave_Somewhat easy,leave_Very difficult,leave_Very easy,mental_health_consequence_No,mental_health_consequence_Yes,phys_health_consequence_No,phys_health_consequence_Yes,coworkers_Some of them,coworkers_Yes,supervisor_Some of them,supervisor_Yes,mental_health_interview_No,mental_health_interview_Yes,phys_health_interview_No,phys_health_interview_Yes,mental_vs_physical_No,mental_vs_physical_Yes
0,37.0,0.0,0,1,0,1,0,0,0,0,1,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,True,True,False,True,False,False,True,False,True,False,True,False,False,True,False,True,False,True,False,False,True,True,False,False,False,False,True
1,44.0,0.0,0,0,0,0,0,0,0,0,1,0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,True,False,False,False
2,32.0,0.0,0,0,0,1,0,0,0,1,1,0,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Fal

In [4]:
# Convert bool to integers
bool_cols = df_ml.select_dtypes(include=["bool"]).columns
df_ml[bool_cols] = df_ml[bool_cols].astype(int)

df_ml.dtypes.value_counts()

for col in df_ml.columns:
    if df_ml[col].dtype == "object":
        unique_vals = set(df_ml[col].dropna().astype(str).unique())
        if unique_vals.issubset({"True", "False"}):
            df_ml[col] = df_ml[col].map({"True": 1, "False": 0})

In [8]:
# Target variable: work interference + treatment
# Create a binary target variable for work interference
# 0 --> no interference, 1 --> interference (rarely/sometimes/often)

df = df_ml.copy()

df["work_interfere_binary"] = (
    df[[
        "work_interfere_Often",
        "work_interfere_Rarely",
        "work_interfere_Sometimes"
    ]].sum(axis=1) > 0
).astype(int)

# Create a combined target variable with three classes:
# 0 --> no interference, 1 --> interference but no treatment, 2 --> interference
df["combined_target"] = None

df.loc[df["work_interfere_binary"] == 0, "combined_target"] = 0

df.loc[
    (df["work_interfere_binary"] == 1) &
    (df["treatment"] == 0),
    "combined_target"
] = 1

df.loc[
    (df["work_interfere_binary"] == 1) &
    (df["treatment"] == 1),
    "combined_target"
] = 2

df["combined_target"] = df["combined_target"].astype(int)

# df["combined_target"].value_counts()
df["combined_target"].value_counts(normalize=True)

combined_target
2    0.482129
1    0.348689
0    0.169182
Name: proportion, dtype: float64

In [19]:
X = df.drop(columns=[
    "combined_target",
    "treatment",
    "work_interfere_Often",
    "work_interfere_Rarely",
    "work_interfere_Sometimes",
    "work_interfere__missing",
    "work_interfere_binary"
])

y = df["combined_target"]

print(X.shape, y.shape)

print(y.isna().sum())
print(y.value_counts())

(1259, 135) (1259,)
0
combined_target
2    607
1    439
0    213
Name: count, dtype: int64


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

Train shape: (1007, 135)
Test shape: (252, 135)

Train class distribution:
combined_target
2    0.482622
1    0.348560
0    0.168818
Name: proportion, dtype: float64


In [20]:
rf_selector = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_selector.fit(X_train, y_train)

selector = SelectFromModel(
    rf_selector,
    threshold="median"
)

selector.fit(X_train, y_train)

X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()]
print("Number of selected features:", len(selected_features))
print("\nSelected features:")
print(selected_features.tolist())

Number of selected features: 68

Selected features:
['Age', 'self_employed', 'family_history', 'remote_work', 'tech_company', 'obs_consequence', 'Gender__missing', 'state__missing', 'self_employed__missing', 'Gender_male', 'Country_Canada', 'Country_France', 'Country_Germany', 'Country_Ireland', 'Country_Netherlands', 'Country_New Zealand', 'Country_United Kingdom', 'Country_United States', 'state_CA', 'state_FL', 'state_GA', 'state_IL', 'state_IN', 'state_MI', 'state_MO', 'state_NY', 'state_OH', 'state_OR', 'state_PA', 'state_TN', 'state_TX', 'state_UT', 'state_VA', 'state_WA', 'state_WI', 'no_employees_100-500', 'no_employees_26-100', 'no_employees_500-1000', 'no_employees_6-25', 'no_employees_More than 1000', 'benefits_No', 'benefits_Yes', 'care_options_Not sure', 'care_options_Yes', 'wellness_program_No', 'wellness_program_Yes', 'seek_help_No', 'seek_help_Yes', 'anonymity_No', 'anonymity_Yes', 'leave_Somewhat difficult', 'leave_Somewhat easy', 'leave_Very difficult', 'leave_Very ea

In [13]:
logreg_hybrid = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

logreg_hybrid.fit(X_train_sel, y_train)
y_pred_hybrid1 = logreg_hybrid.predict(X_test_sel)

print("Hybrid 1: RF Feature Selection -> Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_hybrid1))
print("Macro F1:", f1_score(y_test, y_pred_hybrid1, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_hybrid1, average="weighted"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_hybrid1))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_hybrid1))

Hybrid 1: RF Feature Selection -> Logistic Regression
Accuracy: 0.5079365079365079
Macro F1: 0.4718008620525345
Weighted F1: 0.5313585450571752

Classification Report:

              precision    recall  f1-score   support

           0       0.22      0.44      0.29        43
           1       0.50      0.39      0.44        88
           2       0.77      0.62      0.68       121

    accuracy                           0.51       252
   macro avg       0.50      0.48      0.47       252
weighted avg       0.58      0.51      0.53       252


Confusion Matrix:

[[19 18  6]
 [37 34 17]
 [30 16 75]]


In [14]:
coef_df = pd.DataFrame(
    logreg_hybrid.coef_,
    columns=selected_features,
    index=[f"class_{c}" for c in logreg_hybrid.classes_]
)

coef_df.T.sort_values(by="class_2", key=np.abs, ascending=False).head(15)

,class_0,class_1,class_2
family_history,-0.728084,-0.405907,1.133991
coworkers_Yes,-0.552570,-0.451551,1.004121
state_OH,-0.358074,-0.434719,0.792793
Country_New Zealand,-0.219199,-0.448653,0.667852
care_options_Yes,-0.328352,-0.317703,0.646055
coworkers_Some of them,-0.416382,-0.226531,0.642913
Country_Germany,-0.646506,0.045250,0.601256
state_MO,0.518481,0.065678,-0.584158
no_employees_500-1000,0.612259,-0.037071,-0.575188
state_TX,-0.133688,-0.417552,0.551240


In [21]:
xgb_selector = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_selector.fit(X_train, y_train)

selector_xgb = SelectFromModel(
    xgb_selector,
    threshold="median"
)

selector_xgb.fit(X_train, y_train)

X_train_sel_xgb = selector_xgb.transform(X_train)
X_test_sel_xgb = selector_xgb.transform(X_test)

selected_features_xgb = X_train.columns[selector_xgb.get_support()]
print("Number of selected features:", len(selected_features_xgb))
print(selected_features_xgb.tolist())

Number of selected features: 68
['Age', 'self_employed', 'family_history', 'remote_work', 'tech_company', 'obs_consequence', 'Gender__missing', 'state__missing', 'self_employed__missing', 'Gender_male', 'Country_Belgium', 'Country_Canada', 'Country_France', 'Country_Germany', 'Country_Ireland', 'Country_Netherlands', 'Country_Sweden', 'Country_United Kingdom', 'Country_United States', 'state_CA', 'state_FL', 'state_GA', 'state_IN', 'state_MI', 'state_MN', 'state_MO', 'state_NC', 'state_NY', 'state_OH', 'state_OR', 'state_PA', 'state_TN', 'state_TX', 'state_VA', 'state_WA', 'no_employees_100-500', 'no_employees_26-100', 'no_employees_500-1000', 'no_employees_6-25', 'no_employees_More than 1000', 'benefits_No', 'benefits_Yes', 'care_options_Not sure', 'care_options_Yes', 'wellness_program_No', 'wellness_program_Yes', 'seek_help_No', 'seek_help_Yes', 'anonymity_No', 'anonymity_Yes', 'leave_Somewhat difficult', 'leave_Somewhat easy', 'leave_Very difficult', 'leave_Very easy', 'mental_healt

In [22]:
logreg_hybrid_xgb = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

logreg_hybrid_xgb.fit(X_train_sel_xgb, y_train)
y_pred_hybrid2 = logreg_hybrid_xgb.predict(X_test_sel_xgb)

print("Hybrid 2: XGB Feature Selection -> Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_hybrid2))
print("Macro F1:", f1_score(y_test, y_pred_hybrid2, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_hybrid2, average="weighted"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_hybrid2))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_hybrid2))

Hybrid 2: XGB Feature Selection -> Logistic Regression
Accuracy: 0.49206349206349204
Macro F1: 0.45411268991735376
Weighted F1: 0.5163574509939219

Classification Report:

              precision    recall  f1-score   support

           0       0.20      0.42      0.27        43
           1       0.47      0.35      0.40        88
           2       0.77      0.62      0.68       121

    accuracy                           0.49       252
   macro avg       0.48      0.46      0.45       252
weighted avg       0.57      0.49      0.52       252


Confusion Matrix:

[[18 19  6]
 [40 31 17]
 [30 16 75]]


In [26]:
base_estimators = [
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )),
    ("rf", RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
]

stack_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=2000, random_state=42),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)

stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("Hybrid 3: Stacking")
print("Accuracy:", accuracy_score(y_test, y_pred_stack))
print("Macro F1:", f1_score(y_test, y_pred_stack, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_stack, average="weighted"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_stack))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_stack))

Hybrid 3: Stacking
Accuracy: 0.6190476190476191
Macro F1: 0.4481132075471698
Weighted F1: 0.5676662174303684

Classification Report:

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        43
           1       0.51      0.72      0.59        88
           2       0.73      0.77      0.75       121

    accuracy                           0.62       252
   macro avg       0.41      0.49      0.45       252
weighted avg       0.53      0.62      0.57       252


Confusion Matrix:

[[ 0 33 10]
 [ 1 63 24]
 [ 0 28 93]]


In [24]:
base_estimators = [
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )),
    ("rf", RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
]

stack_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced"),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)

stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print("Hybrid 3: Stacking")
print("Accuracy:", accuracy_score(y_test, y_pred_stack))
print("Macro F1:", f1_score(y_test, y_pred_stack, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_stack, average="weighted"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_stack))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_stack))

Hybrid 3: Stacking
Accuracy: 0.5198412698412699
Macro F1: 0.4808448540706605
Weighted F1: 0.5392782166630092

Classification Report:

              precision    recall  f1-score   support

           0       0.26      0.42      0.32        43
           1       0.41      0.41      0.41        88
           2       0.80      0.64      0.71       121

    accuracy                           0.52       252
   macro avg       0.49      0.49      0.48       252
weighted avg       0.57      0.52      0.54       252


Confusion Matrix:

[[18 21  4]
 [37 36 15]
 [14 30 77]]


In [25]:
results = pd.DataFrame([
    {
        "Model": "RF selection -> Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred_hybrid1),
        "Macro F1": f1_score(y_test, y_pred_hybrid1, average="macro"),
        "Weighted F1": f1_score(y_test, y_pred_hybrid1, average="weighted"),
    },
    {
        "Model": "XGB selection -> Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred_hybrid2),
        "Macro F1": f1_score(y_test, y_pred_hybrid2, average="macro"),
        "Weighted F1": f1_score(y_test, y_pred_hybrid2, average="weighted"),
    },
    {
        "Model": "Stacking (LR + RF + XGB)",
        "Accuracy": accuracy_score(y_test, y_pred_stack),
        "Macro F1": f1_score(y_test, y_pred_stack, average="macro"),
        "Weighted F1": f1_score(y_test, y_pred_stack, average="weighted"),
    }
])

results.sort_values("Macro F1", ascending=False)

,Model,Accuracy,Macro F1,Weighted F1
2,Stacking (LR + RF + XGB),0.519841,0.480845,0.539278
0,RF selection -> Logistic Regression,0.507937,0.471801,0.531359
1,XGB selection -> Logistic Regression,0.492063,0.454113,0.516357
